In [4]:
# --- CONFIGURATION ---
URI = "bolt://localhost:7687"  # Direct local connection
AUTH = ("neo4j", "VzZ@BTLg@JBoG94NzPJg9VJeTHaSRyk2") 
DB_NAME = "neo4j"

In [2]:
# CSV to neo4j
import json
import math
import re
import time
from datetime import datetime
from typing import Dict, Any, List, Tuple
from neo4j import GraphDatabase
import socket
import polars as pl

# --- LOAD DATA WITH POLARS ---
df = pl.scan_csv("../Data/final.csv").collect()

# --- CONFIGURATION UPDATES ---
data = df
START_INDEX = 10001
# END_INDEX = df.height - 1
BATCH_SIZE = 10000
END_INDEX = 200000

TURBO_QUERY = """
UNWIND $batch AS data

// 1. Professional
MERGE (p:Professional {linkedin_id: data.person.id})
SET p.name = data.person.name, 
    p.url = data.person.url, 
    p.about = data.person.about

// 2. Industry
WITH data, p
MERGE (ci:Industry {id: data.person.current_industry_id}) 
    ON CREATE SET ci.name = data.person.current_industry
MERGE (fi:Industry {id: data.person.first_industry_id}) 
    ON CREATE SET fi.name = data.person.first_industry
MERGE (p)-[:CURRENT_INDUSTRY]->(ci)
MERGE (p)-[:FIRST_INDUSTRY]->(fi)

// 3. Experience
WITH data, p
FOREACH (job IN data.jobs |
    MERGE (c:Company {id: job.comp_id}) SET c.name = job.company
    MERGE (jt:JobTitle {id: job.jt_id}) SET jt.name = job.title

    CREATE (exp:Experience {
        id: job.exp_id,
        start_date: job.start_date,
        end_date: job.end_date,
        description: job.description,
        duration_days: job.duration_days
    })
    CREATE (p)-[:HAS_EXPERIENCE]->(exp)
    CREATE (exp)-[:AT_COMPANY]->(c)
    CREATE (exp)-[:ROLE_WAS]->(jt)
)

// 4. Education
WITH data, p
FOREACH (edu IN data.edus |

    MERGE (u:University {id: edu.uni_id}) 
    ON CREATE SET u.name = edu.institution


    MERGE (deg:Degree {id: edu.deg_std_id})
    ON CREATE SET deg.name = edu.deg_std_name

    MERGE (maj:Major {id: edu.maj_std_id})
    ON CREATE SET maj.name = edu.maj_std_name

    CREATE (ed:Education {
        id: data.person.id + "_" + edu.uni_id + "_" + edu.start_date,
        degree_raw: edu.degree_raw,
        major_raw: edu.major_raw,
        end_year: edu.end_year,
        description: edu.description
    })

    CREATE (p)-[:HAS_EDUCATION]->(ed)
    CREATE (ed)-[:AT_UNIVERSITY]->(u)
    CREATE (ed)-[:FOR_DEGREE]->(deg)
    CREATE (ed)-[:IN_MAJOR]->(maj)
)

// 5. Certifications
WITH data, p
FOREACH (cert IN data.certs |
    MERGE (cr:Certification {id: cert.cert_id})
    SET cr.title = cert.title,
        cr.subtitle = cert.subtitle,
        cr.credential_id = cert.credential_id,
        cr.credential_url = cert.credential_url,
        cr.meta = cert.meta,
        cr.issued_date = cert.issued_date
    MERGE (p)-[:HAS_CERTIFICATION]->(cr)
)

WITH data
UNWIND data.job_links AS link
MATCH (s:Experience {id: link.from})
MATCH (t:Experience {id: link.to})
CALL apoc.create.relationship(s, link.type, {}, t) YIELD rel
RETURN count(*)
"""

# --- HELPER FUNCTIONS ---
def extract_meta_date(meta: Any) -> str:
    """Extracts 'Month Year' from strings like 'Issued Jun 2024 Credential ID...'"""
    if not isinstance(meta, str):
        return "N/A"
    
    match = re.search(r"Issued\s+([A-Z][a-z]{2,8})\s+(\d{4})", meta)
    if match:
        month, year = match.groups()
        valid_months = {"Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
                        "January", "February", "March", "April", "May", "June",
                        "July", "August", "September", "October", "November", "December"}
        if month in valid_months:
            return f"{month} {year}"
    return "N/A"

def node_id(text: Any) -> str:
    return re.sub(r'\W+', '_', str(text).lower()).strip('_')

def parse_date(date_str: Any) -> datetime:
    if not isinstance(date_str, str) or date_str.lower() == 'present':
        return datetime(2026, 1, 1)
    for fmt in ('%b %Y', '%Y'):
        try: return datetime.strptime(date_str, fmt)
        except: continue
    return datetime(1900, 1, 1)

def safe_get(row: Dict, key: str, default: Any = "N/A") -> Any:
    val = row.get(key)
    if val is None or (isinstance(val, float) and math.isnan(val)): 
        return default
    return val

def transform_row_to_map(row: Dict) -> Dict:
    def get_json(key):
        val = row.get(key)
        if val is None or val == "": return []
        return json.loads(val) if isinstance(val, str) else val

    person_id = safe_get(row, 'id', 'unknown')

    raw_certs = json.loads(row.get('certifications')) if row.get('certifications') else []
    certs = []
    for i, c in enumerate(raw_certs):
        title = c.get('title', 'Unknown Certificate')
        certs.append({
            'cert_id': f"{person_id}_cert_{i}",
            'title': title,
            'subtitle': c.get('subtitle', 'N/A'),
            'credential_id': c.get('credential_id', 'N/A'),
            'credential_url': c.get('credential_url', 'N/A'),
            'meta': c.get('meta', ''),
            'issued_date': extract_meta_date(c.get('meta'))
        })

    # Process Experience
    raw_jobs = get_json('jobs')
    jobs = []
    for i, job in enumerate(raw_jobs):
        start = parse_date(job.get('start_date'))
        end = parse_date(job.get('end_date'))
        jobs.append({
            'exp_id': f"{person_id}_job_{i}",
            'comp_id': node_id(job.get('company', 'Unknown')),
            'jt_id': node_id(job.get('title', 'Unknown')),
            'ind_id': node_id(job.get('industry_standardized', 'unknown')),
            'company': job.get('company', 'Unknown'),
            'title': job.get('title', 'Unknown'),
            'industry': job.get('industry_standardized', 'Unknown'),
            'description': job.get('description', ''),
            'start_date': job.get('start_date'),
            'end_date': job.get('end_date'),
            'parsed_start': start,
            'parsed_end': end,
            'duration_days': (end - start).days
        })


    jobs.sort(key=lambda x: (x['parsed_start'], -x['duration_days']))

    job_links = []
    for i in range(len(jobs)):
        for j in range(i + 1, len(jobs)):
            current = jobs[i]
            next_job = jobs[j]

            if next_job['parsed_start'] >= current['parsed_end']:
                job_links.append({"from": current['exp_id'], "to": next_job['exp_id'], "type": "FOLLOWED_BY"})
                # Break here to maintain linearity - we found the 'next' step
                break 

            elif next_job['parsed_start'] >= current['parsed_start']:
                # If it also ends before/with the current one, it's a sub-role
                if next_job['parsed_end'] <= current['parsed_end']:
                    job_links.append({"from": current['exp_id'], "to": next_job['exp_id'], "type": "HAPPENED_DURING"})
                    # We don't break; there might be another sub-role or a following role
                else:
                    job_links.append({"from": current['exp_id'], "to": next_job['exp_id'], "type": "PITCHED_INTO"})
                    break

    return {
        "person": {
            "id": person_id,
            "name": safe_get(row, 'name', 'N/A'),
            "url": safe_get(row, 'url', 'N/A'),
            "about": safe_get(row, 'about', ''),
            "city": safe_get(row, 'city', 'N/A'),
            "followers": safe_get(row, 'followers', 0),
            "connections": safe_get(row, 'connections', 0),
            "current_industry": safe_get(row, 'current_industry'),
            "current_industry_id": node_id(safe_get(row, 'current_industry', 'unknown')),
            "first_industry": safe_get(row, 'first_industry'),
            "first_industry_id": node_id(safe_get(row, 'first_industry', 'unknown'))
        },
        "certs": certs,
        "jobs": jobs,
        "job_links": job_links,
        "edus": [{
            "uni_id": node_id(e.get('university', 'Unknown')),
            "institution": e.get('university', 'Unknown'),
            "degree_raw": e.get('degree_user_input', 'N/A'),
            "major_raw": e.get('major_user_input', 'N/A'),
            "deg_std_id": node_id(e.get('degree_standardized', 'n_a')),
            "deg_std_name": e.get('degree_standardized'),
            "maj_std_id": node_id(e.get('major_standardized', 'n_a')),
            "maj_std_name": e.get('major_standardized'),
            "start_date": str(e.get('start_date', 'N/A')),
            "end_date": str(e.get('end_date', 'N/A')),
            "description": e.get('description', '')
        } for e in get_json('educations')]
    }

if __name__ == "__main__":
    print(f"🚀 Starting Debug Import (Fixing Transaction Scope)...")
    total_start = time.perf_counter()

    try:
        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            driver.verify_connectivity()
            
            for offset in range(START_INDEX, END_INDEX + 1, BATCH_SIZE):
                length = min(BATCH_SIZE, (END_INDEX + 1) - offset)
                
                # --- METRIC 1: PYTHON PREP ---
                prep_start = time.perf_counter()
                batch_dicts = df.slice(offset, length).to_dicts()
                current_batch = [transform_row_to_map(row) for row in batch_dicts]
                prep_time = time.perf_counter() - prep_start

                if current_batch:
                    # --- METRIC 2: DATABASE EXECUTION ---
                    def execute_and_measure(tx, batch):
                        start = time.perf_counter()
                        # Running the query
                        result = tx.run(TURBO_QUERY, batch=batch)
                        # Consuming metadata inside the transaction scope
                        summary = result.consume() 
                        end = time.perf_counter()
                        return end - start, summary

                    with driver.session(database=DB_NAME) as session:
                        db_time, summary = session.execute_write(execute_and_measure, current_batch)
                    
                    print(f"📦 Batch {offset}-{offset + length - 1}")
                    print(f"   └─ Neo4j Write : {db_time:.3f}s, prep time: {prep_time:.3f}")

    except Exception as e:
        print(f"❌ Critical Error: {e}")

🚀 Starting Debug Import (Fixing Transaction Scope)...
📦 Batch 10001-20000
   └─ Neo4j Write : 17.575s, prep time: 1.564
📦 Batch 20001-30000
   └─ Neo4j Write : 12.435s, prep time: 1.234
📦 Batch 30001-40000
   └─ Neo4j Write : 12.578s, prep time: 1.517
📦 Batch 40001-50000
   └─ Neo4j Write : 12.503s, prep time: 1.340
📦 Batch 50001-60000
   └─ Neo4j Write : 12.272s, prep time: 1.310
📦 Batch 60001-70000
   └─ Neo4j Write : 12.023s, prep time: 1.318
📦 Batch 70001-80000
   └─ Neo4j Write : 12.299s, prep time: 1.379
📦 Batch 80001-90000
   └─ Neo4j Write : 12.269s, prep time: 1.459
📦 Batch 90001-100000
   └─ Neo4j Write : 12.242s, prep time: 1.493
📦 Batch 100001-110000
   └─ Neo4j Write : 12.700s, prep time: 1.314
📦 Batch 110001-120000
   └─ Neo4j Write : 12.513s, prep time: 1.499
📦 Batch 120001-130000
   └─ Neo4j Write : 12.295s, prep time: 1.297
📦 Batch 130001-140000
   └─ Neo4j Write : 12.334s, prep time: 1.323
📦 Batch 140001-150000
   └─ Neo4j Write : 12.034s, prep time: 1.315
📦 Batch 150

In [7]:
# Embed the data
import ollama
from neo4j import GraphDatabase
from tqdm import tqdm
import time

# --- CONFIGURATION ---
EMBED_MODEL = "mxbai-embed-large"
MAX_CHAR_LIMIT = 300

driver = GraphDatabase.driver(URI, auth=AUTH)

def run_embedding_task(label, query):
    with driver.session() as session:
        # Measure query time
        query_start = time.time()
        nodes = session.run(query).data()
        query_duration = time.time() - query_start
        
        if not nodes:
            print(f"[-] No pending {label} nodes found.")
            return

        total_nodes = len(nodes)
        print(f"[*] Found {total_nodes} {label} nodes. (Query: {query_duration:.2f}s)")
        
        total_embedding_time = 0
        pbar = tqdm(nodes, desc=f"Processing {label}", unit="node")
        
        for i, node in enumerate(pbar):
            node_id = str(node.get('id') or 'UNKNOWN_ID')
            raw_text = node.get('text') or ""
            
            try:

                if len(raw_text) > MAX_CHAR_LIMIT:
                    processed_text = raw_text[:MAX_CHAR_LIMIT] + "..."
                else:
                    processed_text = raw_text

                node_start = time.time()
                
                # Generate embedding
                res = ollama.embeddings(model=EMBED_MODEL, prompt=processed_text)
                
                # Update Neo4j 
                # Note: We must match by the SAME property used in the RETURN alias
                # For Professional, this will be linkedin_id (see query below)
                match_prop = "linkedin_id" if label == "Professional" else "id"
                
                session.run(f"""
                    MATCH (n:{label} {{{match_prop}: $id}})
                    SET n.embedding = $vec, n.embedding_summary = $text
                """, id=node.get('id'), vec=res['embedding'], text=raw_text)
                
                # Stats tracking
                node_duration = time.time() - node_start
                total_embedding_time += node_duration
                avg_time = total_embedding_time / (i + 1)
                
                pbar.set_postfix({
                    "id": node_id[-8:], 
                    "len": len(raw_text),
                    "curr_s": f"{node_duration:.2f}",
                    "avg_s": f"{avg_time:.2f}"
                })
                
            except Exception as e:
                # DEBUG STRATEGY: Print length details on error
                print(f"\n" + "!"*30)
                print(f"FAILED: {label} ID: {node_id}")
                print(f"ERROR: {e}")
                print(f"LENGTH: {len(raw_text)} characters")
                print(f"SNIPPET: {raw_text[:100]}...")
                print("!"*30 + "\n")
                continue

        print(f"[✓] Completed {label}. Avg: {total_embedding_time/total_nodes:.2f}s/node\n")

# --- TASK QUERIES ---

# 1. Experience:
experience_query = """
MATCH (e:Experience)-[:ROLE_WAS]->(j:JobTitle)
MATCH (e)-[:AT_COMPANY]->(c:Company)
WHERE e.embedding IS NULL
RETURN e.id as id, 
    'Job: ' + coalesce(j.name, 'unknown job title') + 
    ' at ' + coalesce(c.name, 'unknown company') + 
    '. Description: ' + coalesce(e.description, 'unknown description') + 
    '. Period: ' + coalesce(e.start_date, 'unknown start date') + ' to ' + coalesce(e.end_date, 'unknown end date') + 
    '. Duration: ' + coalesce(toString(e.duration_days), '0') + ' days.' as text
"""

# 2. Education:
education_query = """
MATCH (ed:Education)
WHERE ed.embedding IS NULL
OPTIONAL MATCH (ed)-[:FOR_DEGREE]->(d:Degree)
OPTIONAL MATCH (ed)-[:IN_MAJOR]->(m:Major)
OPTIONAL MATCH (ed)-[:AT_UNIVERSITY]->(u:University)
RETURN ed.id as id, 
    'University: ' + coalesce(u.name, 'unknown university') + 
    '. Degree: ' + coalesce(ed.degree_raw, d.name, 'unknown degree') + 
    '. Major: ' + coalesce(ed.major_raw, m.name, 'unknown major') + '.' as text
"""

# 3. Certification:
certification_query = """
MATCH (c:Certification)
WHERE c.embedding IS NULL
RETURN c.id as id, 
    'Certification: ' + coalesce(c.title, 'unknown certification title') + 
    '. Issued by: ' + coalesce(c.subtitle, 'unknown issuer') + 
    '. Date: ' + coalesce(c.issued_date, 'unknown issued date') + '.' as text
"""

# 4. Professional:
professional_query = """
MATCH (p:Professional)
WHERE p.embedding IS NULL
OPTIONAL MATCH (p)-[:CURRENT_INDUSTRY]->(ci:Industry)
OPTIONAL MATCH (p)-[:FIRST_INDUSTRY]->(fi:Industry)
WITH p, ci, fi,
     COUNT { (p)-[:HAS_EXPERIENCE]->() } as expCount,
     COUNT { (p)-[:HAS_EDUCATION]->() } as eduCount
RETURN p.linkedin_id as id, 
    'Name: ' + coalesce(p.name, 'unknown name') + 
    '. Summary: ' + coalesce(p.about, 'unknown summary') + 
    '. Current Industry: ' + coalesce(ci.name, 'unknown current industry') + 
    '. First Industry: ' + coalesce(fi.name, 'unknown first industry') + 
    '. Experience Count: ' + toString(expCount) + 
    '. Education Count: ' + toString(eduCount) + '.' as text
"""

# --- EXECUTION ---
tasks = [
    ("Education", education_query),
    ("Certification", certification_query),
    ("Professional", professional_query),
    ("Experience", experience_query),
]

overall_start = time.time()

for label, query in tasks:
    run_embedding_task(label, query)

driver.close()
total_runtime = time.time() - overall_start
print("-" * 30)
print(f"All embeddings complete! Total Script Runtime: {total_runtime:.2f}s")

[-] No pending Education nodes found.
[-] No pending Certification nodes found.
[*] Found 2 Professional nodes. (Query: 0.08s)


Processing Professional: 100%|██████████| 2/2 [00:00<00:00, 22.96node/s, id=110525ba, len=892, curr_s=0.04, avg_s=0.04]


[✓] Completed Professional. Avg: 0.04s/node

[-] No pending Experience nodes found.
------------------------------
All embeddings complete! Total Script Runtime: 1.47s
